# Simple RAG with Filtering + Query Transformations

## 적용 기법
- **Metadata Filtering** : cate_depth1 기반 장르 필터
- **Query Transformations** : Sub-query Decomposition → 서브쿼리별 검색 → RRF 결합

## 변경 사항 (simple_rag_with_filtering 대비)
- `simple_rag_with_filter_node` → `query_transform_rag_node` 로 교체
- 사용자 프로파일을 서브쿼리로 분해하여 각각 검색 후 RRF로 결합
- 나머지 구조 (설치, 초기화, genre 노드, llm 노드) 동일 유지

## 0. 설치 및 초기화

In [ ]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

In [ ]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver

In [ ]:
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/state/state_v1.ipynb
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/db/qdrant.py
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/embedding/embedder.py

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"]    = userdata.get('OPENAI_API_KEY')
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')
os.environ["QDRANT_API_KEY"]    = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"]        = userdata.get('QDRANT_URL')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

embedder = LocalEmbedder("BAAI/bge-m3")
db       = QdrantDB(vector_size=1024)
llm      = ChatOpenAI(model="gpt-4o-mini")
# llm    = ChatClovaX(model="HCX-005", temperature=0)

## 1. 장르 추출 노드 (simple_rag_with_filtering과 동일)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client.models import Filter, FieldCondition, MatchAny
import json

CATEGORY_LIST = ["소설", "대학교재/전문서적", "어린이", "수험서/자격증", "시/에세이", "종교", "유아", "만화",
                 "사회/정치", "경제/경영", "인문", "예술/대중문화", "국어/외국어", "고등학교 참고서",
                 "자기계발", "초등학교 참고서", "건강/취미", "컴퓨터/IT", "역사", "자연/과학",
                 "청소년", "중학교 참고서", "가정/요리", "여행", "잡지", "전집", "외국도서"]

genre_prompt = ChatPromptTemplate.from_template("""
사용자 프로파일을 보고 아래 카테고리 목록에서 적합한 것을 2개만 선택하세요.
목록에 없는 값은 절대 반환하지 마세요.

카테고리 목록: {category_list}
사용자 프로파일: {summary}

JSON으로만 반환: {{"categories": ["소설"]}}
""")

def extract_genre_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    chain = genre_prompt | llm
    response = chain.invoke({
        "category_list": CATEGORY_LIST,
        "summary": summary
    })
    try:
        categories = json.loads(response.content)["categories"]
    except (json.JSONDecodeError, KeyError):
        categories = []
    print(f"추출된 장르: {categories}")
    return {"genre_filter": categories}

## 2. Sub-query Decomposition 함수

사용자 프로파일을 2~4개의 서브쿼리로 분해

In [ ]:
from langchain.prompts import PromptTemplate

subquery_decomposition_template = """당신은 도서 추천 시스템의 AI 어시스턴트입니다.
사용자 프로파일을 분석하여 도서 검색에 활용할 2~4개의 핵심 서브쿼리로 분해하세요.
각 서브쿼리는 독서 목적, 선호 장르, 현재 상태, 난이도 등 서로 다른 측면을 담아야 합니다.

사용자 프로파일: {summary}

출력 형식 (번호와 텍스트만, 다른 텍스트 없이):
1. [서브쿼리 1]
2. [서브쿼리 2]
3. [서브쿼리 3]"""

subquery_decomposition_prompt = PromptTemplate(
    input_variables=["summary"],
    template=subquery_decomposition_template
)

subquery_decomposer_chain = subquery_decomposition_prompt | llm

def decompose_query(summary: str) -> list:
    """
    사용자 프로파일을 서브쿼리로 분해
    Returns: List[str]
    """
    response = subquery_decomposer_chain.invoke({"summary": summary}).content
    sub_queries = [
        q.strip().lstrip("1234567890. ")
        for q in response.split("\n")
        if q.strip() and q.strip()[0].isdigit()
    ]
    return sub_queries

## 3. RRF (Reciprocal Rank Fusion) 함수

In [ ]:
def reciprocal_rank_fusion(results_list: list, k: int = 60) -> list:
    """
    여러 검색 결과를 RRF로 결합

    Args:
        results_list : 각 서브쿼리별 검색 결과 리스트
        k            : RRF 상수 (기본값 60)

    Returns:
        ISBN 기준으로 정렬된 결과 딕셔너리 리스트
    """
    scores = {}
    payloads = {}

    for results in results_list:
        for rank, r in enumerate(results):
            isbn = r.payload.get("isbn", "")
            if isbn:
                scores[isbn]   = scores.get(isbn, 0) + 1 / (k + rank + 1)
                payloads[isbn] = r.payload  # 마지막 payload 저장

    # 점수 내림차순 정렬
    sorted_isbns = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [payloads[isbn] for isbn, _ in sorted_isbns]

## 4. Query Transformations RAG 노드

`simple_rag_with_filter_node` 대체 — 기본 구조(필터링) 유지 + Sub-query Decomposition 추가

In [ ]:
def query_transform_rag_node(state: CRSState) -> dict:
    summary    = state.get("summary", "")
    categories = state.get("genre_filter", [])

    # ── 1. Sub-query Decomposition ──────────────────────────
    sub_queries = decompose_query(summary)
    print(f"\n서브쿼리 목록:")
    for i, q in enumerate(sub_queries, 1):
        print(f"  {i}. {q}")

    # ── 2. 장르 필터 구성 (simple_rag_with_filtering과 동일) ──
    query_filter = None
    if categories:
        query_filter = Filter(
            must=[FieldCondition(
                key="cate_depth1",
                match=MatchAny(any=categories)
            )]
        )

    # ── 3. 서브쿼리별 검색 ──────────────────────────────────
    all_results = []
    for sub_query in sub_queries:
        query_vector = embedder.embed(sub_query)
        if query_filter:
            results = db.search_with_filter(
                "books_v1",
                query_vector,
                query_filter=query_filter,
                limit=10,
                threshold=0.5,
            )
        else:
            results = db.search("books_v1", query_vector, limit=10, threshold=0.5)
        all_results.append(results)

    # ── 4. RRF 결합 ─────────────────────────────────────────
    merged_payloads = reciprocal_rank_fusion(all_results)

    # ── 5. 결과 정리 ─────────────────────────────────────────
    retrieved_books = [
        {
            "isbn":       p.get("isbn"),
            "title":      p.get("title"),
            "author":     p.get("author"),
            "book_intro": p.get("book_intro"),
            "cate_depth1": p.get("cate_depth1"),
        }
        for p in merged_payloads[:10]  # 상위 10개 LLM에 전달
    ]

    print(f"\n최종 검색 결과: {len(retrieved_books)}권")
    return {"retrieved_books": retrieved_books}

## 5. LLM 노드 (simple_rag_with_filtering과 동일)

In [ ]:
def rag_llm_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    books   = state["retrieved_books"]

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"장르: {b['cate_depth1']}\n"
        f"소개: {b['book_intro']}"
        for b in books
    ])

    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 추천 이유는 반드시 사용자 프로파일의 독서 목적, 성향, 상황과 연결해서 작성하세요.
- 장르는 반드시 [검색된 도서 목록]의 장르 값을 그대로 사용하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "cate1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "cate1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "cate1": "장르", "reason": "추천 이유 2~3문장"}}
]
""")

    chain    = rag_prompt | llm
    response = chain.invoke({"context": context, "summary": summary})

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        recommendations = response.content

    return {
        "messages":       [AIMessage(content=response.content)],
        "recommendations": recommendations
    }

## 6. 그래프 구성 및 실행

In [ ]:
graph_test = StateGraph(CRSState)
graph_test.add_node("genre", extract_genre_node)
graph_test.add_node("rag",   query_transform_rag_node)  # ← 변경된 노드
graph_test.add_node("llm",   rag_llm_node)
graph_test.add_edge(START,   "genre")
graph_test.add_edge("genre", "rag")
graph_test.add_edge("rag",   "llm")
graph_test.add_edge("llm",   END)
app_test = graph_test.compile()

# TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."

result = app_test.invoke({
    "messages":        [HumanMessage(content=TEST_SUMMARY)],
    "summary":         TEST_SUMMARY,
    "retrieved_books": [],
    "recommendations": [],
    "genre_filter":    [],
})

print("\n📚 추천 결과:")
for book in result["recommendations"]:
    print(f"\n제목: {book['title']}")
    print(f"저자: {book['author']}")
    print(f"장르: {book['cate1']}")
    print(f"추천 이유: {book['reason']}")

## 7. DeepEval 평가

In [ ]:
import logging
logging.getLogger("deepeval").setLevel(logging.WARNING)

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=TEST_SUMMARY,
        actual_output="\n".join([
            f"[{b['title']}] {b['reason']}"
            for b in result["recommendations"]
        ]),
        retrieval_context=[
            f"[{b['title']}] {b['book_intro']}"
            for b in result["retrieved_books"]
        ]
    )
]

evaluate(test_cases, [
    AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
    FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
])